# English → Hindi Machine Translation: Does More (Synthetic) Data Help?

**Experiment**: Compare fine-tuning `Helsinki-NLP/opus-mt-en-hi` on 3000 parallel pairs (baseline having 1000 duplicated sentences from data itself) vs 3000 pairs (baseline + 1000 LLM-paraphrased English sentences with same Hindi targets).

**Data Split**: 2000 train / 250 validation / 250 test (total 2500 samples, seed=42)

**Workflow**:
1. Load data → split → train baseline
2. Export 1000 samples → paraphrase externally via llm
3. Load paraphrased CSV → build augmented training set → retrain → compare

**Runtime**: Set to `T4 GPU` before running *(Runtime → Change runtime type → T4 GPU)*

## Setup

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu rouge-score sentencepiece sentence-transformers


In [ ]:
import os
import time
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from sentence_transformers import SentenceTransformer
import evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Load & Prepare Data

Sample 2500 pairs from the full IIT-B corpus (seed=42) and split:
- **Train**: 2000 pairs
- **Validation**: 250 pairs (used during training for early stopping)
- **Test**: 250 pairs (held-out for final evaluation)

In [ ]:
from datasets import load_dataset

raw = load_dataset("cfilt/iitb-english-hindi", split="train")
print(f"Full dataset size: {len(raw)}")

In [ ]:
import pandas as pd

SEED = 42

# Step 1: Sample a larger subset
sampled_large = raw.shuffle(seed=SEED).select(range(20000))

# Step 2: Convert to pandas
df = pd.DataFrame(list(sampled_large["translation"]))

# Step 3: Cleaning (vectorized)

# Remove nulls
df = df.dropna(subset=["en", "hi"])

# Strip whitespace
df["en"] = df["en"].str.strip()
df["hi"] = df["hi"].str.strip()

# Word count filter (> 3 words)
df = df[
    (df["en"].str.split().str.len() > 3) &
    (df["hi"].str.split().str.len() > 3)
]

# Remove numbers and noisy patterns
df = df[
    ~df["en"].str.contains(r"\d|\{.*?\}|\$|%", regex=True) &
    ~df["hi"].str.contains(r"\d|\{.*?\}|\$|%", regex=True)
]

# Step 4: Take final 2500
df_clean = df.sample(n=2500, random_state=SEED).reset_index(drop=True)

# Step 5: Split
train_df = df_clean.iloc[:2000]
val_df   = df_clean.iloc[2000:2250]
test_df  = df_clean.iloc[2250:2500]

train_en, train_hi = train_df["en"].tolist(), train_df["hi"].tolist()
val_en, val_hi     = val_df["en"].tolist(), val_df["hi"].tolist()
test_en, test_hi   = test_df["en"].tolist(), test_df["hi"].tolist()

print(f"Train: {len(train_en)} | Validation: {len(val_en)} | Test: {len(test_en)}")
print("\nSample pair:")
print("EN:", train_en[0])
print("HI:", train_hi[0])

## Export Samples for Paraphrasing

Select 1000 random training samples (fixed seed) and export to CSV.
These will be paraphrased externally using llm

In [ ]:
# Select 1000 random samples from training data for paraphrasing
PARAPHRASE_SEED = 42
PARAPHRASE_COUNT = 1000

rng = np.random.RandomState(PARAPHRASE_SEED)
para_indices = rng.choice(len(train_en), size=PARAPHRASE_COUNT, replace=False)
para_indices.sort()

samples_en = [train_en[i] for i in para_indices]
samples_hi = [train_hi[i] for i in para_indices]

# Save to CSV for external paraphrasing
df_export = pd.DataFrame({"original_en": samples_en, "hi": samples_hi})
df_export.to_csv("samples_to_paraphrase.csv", index=False)
print(f"Exported {len(df_export)} samples to samples_to_paraphrase.csv")
print("\nFirst 3 samples:")
df_export.head(3)

In [ ]:
#duplication
original_train_en = train_en.copy()
original_train_hi = train_hi.copy()

baseline_train_en = original_train_en + samples_en
baseline_train_hi = original_train_hi + samples_hi


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(en_list, hi_list):
    return Dataset.from_dict({"en": en_list, "hi": hi_list})

def tokenize(batch):
    model_inputs = tokenizer(
        batch["en"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        text_target=batch["hi"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    # Replace padding token id with -100 so loss ignores them
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in lab]
        for lab in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

train_dataset_base = make_hf_dataset(baseline_train_en, baseline_train_hi).map(tokenize, batched=True)
val_dataset         = make_hf_dataset(val_en, val_hi).map(tokenize, batched=True)
test_dataset        = make_hf_dataset(test_en, test_hi).map(tokenize, batched=True)

for ds in [train_dataset_base, val_dataset, test_dataset]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done.")
print(f"  train : {len(train_dataset_base)}")
print(f"  val   : {len(val_dataset)}")
print(f"  test  : {len(test_dataset)}")


## Baseline Fine-tuning (3000 pairs)

In [ ]:
bleu_metric  = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
chrf_metric  = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,   skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,  skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu  = bleu_metric.compute(predictions=decoded_preds, references=[[r] for r in decoded_labels])
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    chrf  = chrf_metric.compute(predictions=decoded_preds,  references=[[r] for r in decoded_labels])

    return {
        "bleu":    round(bleu["score"], 2),
        "rouge1":  round(rouge["rouge1"], 4),
        "rougeL":  round(rouge["rougeL"], 4),
        "chrf":    round(chrf["score"],  2),
    }

In [ ]:
def get_training_args(output_dir):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LEN,
        generation_num_beams=4,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
        logging_steps=50,
        report_to="none",
    )

def train_and_evaluate(train_ds, val_ds, test_ds, output_dir):
    # Train on train_ds, use val_ds for early stopping, evaluate on test_ds.
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
    args = get_training_args(output_dir)

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Final evaluation on held-out test set
    test_results = trainer.evaluate(test_ds)
    print("\nTest set results:", test_results)
    return trainer, test_results


In [ ]:
print("=== Training Baseline Model (3000 pairs) ===")
print("  Training on: 3000 pairs | Validation: 250 pairs | Test: 250 pairs\n")
baseline_trainer, baseline_results = train_and_evaluate(
    train_dataset_base, val_dataset, test_dataset, output_dir="./baseline_model"
)

## Load Paraphrased Data & Build Augmented Dataset

Load the CSV generated by `generate_paraphrases.py` and combine with the original training data.

In [ ]:
# Load the paraphrased CSV
PARAPHRASE_CSV = Path("/content/paraphrased_samples.csv")  # adjust path if needed
FILTERED_PARAPHRASE_CSV = Path("/content/paraphrased_samples_filtered.csv")
PARAPHRASE_MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PARAPHRASE_BATCH_SIZE = 32
PARAPHRASE_RETURNS = 3
PARAPHRASE_TARGET = 1000
MIN_SIMILARITY = 0.88
MAX_SIMILARITY = 0.995
MIN_TOKEN_CHANGE = 0.12
MIN_LENGTH_RATIO = 0.75
MAX_LENGTH_RATIO = 1.35

def normalize_text(text):
    return " ".join(str(text).replace("\n", " ").split())

def token_change_ratio(source, candidate):
    source_tokens = set(normalize_text(source).lower().split())
    candidate_tokens = set(normalize_text(candidate).lower().split())
    if not source_tokens:
        return 0.0
    return 1.0 - (len(source_tokens & candidate_tokens) / len(source_tokens))

def score_paraphrase_frame(df_frame):
    df_frame = df_frame.copy()
    df_frame["original_en"] = df_frame["original_en"].fillna("").map(normalize_text)
    df_frame["paraphrased_en"] = df_frame["paraphrased_en"].fillna("").map(normalize_text)
    df_frame["hi"] = df_frame["hi"].fillna("").map(normalize_text)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)
    original_embeddings = embedder.encode(
        df_frame["original_en"].tolist(),
        batch_size=64,
        convert_to_tensor=True,
        show_progress_bar=True,
    )
    paraphrase_embeddings = embedder.encode(
        df_frame["paraphrased_en"].tolist(),
        batch_size=64,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    similarities = torch.nn.functional.cosine_similarity(
        original_embeddings, paraphrase_embeddings
    ).cpu().numpy()

    df_frame["semantic_similarity"] = np.round(similarities, 4)
    df_frame["token_change_ratio"] = [
        round(token_change_ratio(src, cand), 4)
        for src, cand in zip(df_frame["original_en"], df_frame["paraphrased_en"])
    ]
    df_frame["length_ratio"] = [
        round(len(cand.split()) / max(len(src.split()), 1), 4)
        for src, cand in zip(df_frame["original_en"], df_frame["paraphrased_en"])
    ]
    df_frame["quality_score"] = np.round(
        df_frame["semantic_similarity"] + 0.2 * df_frame["token_change_ratio"].clip(upper=0.4),
        4,
    )
    return df_frame

def build_paraphrase_csv(df_source, output_path):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    para_tokenizer = AutoTokenizer.from_pretrained(PARAPHRASE_MODEL_NAME)
    para_model = AutoModelForSeq2SeqLM.from_pretrained(PARAPHRASE_MODEL_NAME).to(device)
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device=device)

    source_rows = df_source[["original_en", "hi"]].copy()
    source_rows["original_en"] = source_rows["original_en"].map(normalize_text)

    records = []
    source_texts = source_rows["original_en"].tolist()
    hindi_texts = source_rows["hi"].tolist()

    for start in range(0, len(source_texts), PARAPHRASE_BATCH_SIZE):
        batch_texts = source_texts[start:start + PARAPHRASE_BATCH_SIZE]
        prompts = [f"paraphrase: {text} </s>" for text in batch_texts]
        encoded = para_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LEN,
        ).to(device)

        with torch.no_grad():
            outputs = para_model.generate(
                **encoded,
                max_length=MAX_INPUT_LEN,
                num_beams=4,
                num_return_sequences=PARAPHRASE_RETURNS,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.9,
                repetition_penalty=1.2,
            )

        decoded = para_tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for offset, source_text in enumerate(batch_texts):
            candidates = [
                normalize_text(decoded[offset * PARAPHRASE_RETURNS + candidate_idx])
                for candidate_idx in range(PARAPHRASE_RETURNS)
            ]
            candidates = [
                cand for cand in dict.fromkeys(candidates)
                if cand and cand.lower() != source_text.lower()
            ]
            if not candidates:
                candidates = [source_text]

            source_embedding = embedder.encode(source_text, convert_to_tensor=True)
            candidate_embeddings = embedder.encode(candidates, convert_to_tensor=True)
            similarities = torch.nn.functional.cosine_similarity(
                source_embedding.unsqueeze(0), candidate_embeddings
            ).cpu().tolist()

            ranked = []
            for candidate_text, similarity in zip(candidates, similarities):
                candidate_length_ratio = len(candidate_text.split()) / max(len(source_text.split()), 1)
                candidate_change_ratio = token_change_ratio(source_text, candidate_text)
                quality_score = similarity + 0.2 * min(candidate_change_ratio, 0.4)
                is_valid = (
                    similarity >= MIN_SIMILARITY
                    and similarity <= MAX_SIMILARITY
                    and candidate_change_ratio >= MIN_TOKEN_CHANGE
                    and MIN_LENGTH_RATIO <= candidate_length_ratio <= MAX_LENGTH_RATIO
                )
                ranked.append({
                    "original_en": source_text,
                    "paraphrased_en": candidate_text,
                    "hi": hindi_texts[start + offset],
                    "semantic_similarity": round(similarity, 4),
                    "token_change_ratio": round(candidate_change_ratio, 4),
                    "length_ratio": round(candidate_length_ratio, 4),
                    "quality_score": round(quality_score, 4),
                    "used_fallback": not is_valid,
                    "is_valid": is_valid,
                })

            ranked.sort(key=lambda row: (row["is_valid"], row["quality_score"]), reverse=True)
            records.append(ranked[0])

    df_built = pd.DataFrame(records)
    df_built.to_csv(output_path, index=False)
    return df_built

if PARAPHRASE_CSV.exists():
    df_para = pd.read_csv(PARAPHRASE_CSV)
    if "original_en" not in df_para.columns:
        df_para["original_en"] = df_export["original_en"]
    missing_quality_cols = {
        "semantic_similarity",
        "token_change_ratio",
        "length_ratio",
        "quality_score",
    } - set(df_para.columns)
    if missing_quality_cols:
        df_para = score_paraphrase_frame(df_para)
        df_para.to_csv(PARAPHRASE_CSV, index=False)
else:
    df_para = build_paraphrase_csv(df_export, PARAPHRASE_CSV)

print(f"Loaded {len(df_para)} paraphrased samples from {PARAPHRASE_CSV}")
print("\nSample paraphrases:")
for i in [0, 5, 10]:
    if i < len(df_para):
        print(f"  Original    : {df_para.iloc[i]['original_en']}")
        print(f"  Paraphrased : {df_para.iloc[i]['paraphrased_en']}")
        print(f"  Similarity  : {df_para.iloc[i]['semantic_similarity']}")
        print(f"  Hindi       : {df_para.iloc[i]['hi']}")
        print()


In [ ]:
# Build augmented dataset: original 2000 + filtered paraphrased samples
df_para["paraphrased_en"] = df_para["paraphrased_en"].fillna("").astype(str).map(normalize_text)
df_para["original_en"] = df_para["original_en"].fillna("").astype(str).map(normalize_text)
df_para["hi"] = df_para["hi"].fillna("").astype(str).map(normalize_text)

quality_mask = (
    df_para["semantic_similarity"].between(MIN_SIMILARITY, MAX_SIMILARITY)
    & (df_para["token_change_ratio"] >= MIN_TOKEN_CHANGE)
    & df_para["length_ratio"].between(MIN_LENGTH_RATIO, MAX_LENGTH_RATIO)
    & (df_para["paraphrased_en"].str.lower() != df_para["original_en"].str.lower())
)

df_para_selected = (
    df_para[quality_mask]
    .sort_values("quality_score", ascending=False)
    .drop_duplicates(subset=["paraphrased_en"])
    .head(PARAPHRASE_TARGET)
    .copy()
)

if len(df_para_selected) < 600:
    df_para_selected = (
        df_para.sort_values("quality_score", ascending=False)
        .drop_duplicates(subset=["paraphrased_en"])
        .head(700)
        .copy()
    )

df_para_selected.to_csv(FILTERED_PARAPHRASE_CSV, index=False)

paraphrased_en = df_para_selected["paraphrased_en"].tolist()
paraphrased_hi = df_para_selected["hi"].tolist()

aug_en = original_train_en + paraphrased_en
aug_hi = original_train_hi + paraphrased_hi
augmented_pair_count = len(aug_en)

print(f"Selected {len(df_para_selected)} high-quality paraphrases")
print(f"Filtered CSV saved to {FILTERED_PARAPHRASE_CSV}")
print(f"Augmented training set size: {augmented_pair_count}")

train_dataset_aug = make_hf_dataset(aug_en, aug_hi).map(tokenize, batched=True)
train_dataset_aug.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


## Augmented Fine-tuning (3000 pairs)

In [ ]:
print(f"=== Training Augmented Model ({augmented_pair_count} pairs) ===")
print(f"  Training on: {augmented_pair_count} pairs | Validation: {len(val_en)} pairs | Test: {len(test_en)} pairs\n")
augmented_trainer, augmented_results = train_and_evaluate(
    train_dataset_aug, val_dataset, test_dataset, output_dir="./augmented_model"
)


## Comparison & Analysis

In [ ]:
# Metrics comparison table (test set results)
metrics = ["eval_bleu", "eval_rouge1", "eval_rougeL", "eval_chrf"]
labels  = ["BLEU", "ROUGE-1", "ROUGE-L", "chrF"]

b_scores = [baseline_results.get(m, 0) for m in metrics]
a_scores = [augmented_results.get(m, 0) for m in metrics]

baseline_label = f"Baseline ({len(baseline_train_en)})"
augmented_label = f"Augmented ({augmented_pair_count})"

df = pd.DataFrame({
    "Metric": labels,
    baseline_label: b_scores,
    augmented_label: a_scores,
    "Δ (improvement)": [round(a - b, 4) for a, b in zip(a_scores, b_scores)],
})
print("Test Set Results\n")
print(df.to_string(index=False))


In [ ]:
# Bar chart comparison
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, b_scores, width, label="Baseline (2000)",  color="#4C72B0")
bars2 = ax.bar(x + width/2, a_scores, width, label="Augmented (3000)", color="#DD8452")

ax.set_title("Baseline vs Augmented Model — Test Set Metrics", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Score")
ax.legend()
ax.bar_label(bars1, fmt="%.2f", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.2f", padding=3, fontsize=9)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150)
plt.show()

In [ ]:
# Qualitative comparison on sample test sentences
def translate(model, sentences, batch_size=8):
    model.eval()
    all_outputs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_INPUT_LEN).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=MAX_TARGET_LEN)
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_outputs.extend(decoded)
    return all_outputs

# Pick 15 test sentences for qualitative review
sample_indices = list(range(15))
sample_en = [test_en[i] for i in sample_indices]
sample_hi = [test_hi[i] for i in sample_indices]

baseline_model  = baseline_trainer.model
augmented_model = augmented_trainer.model

base_preds = translate(baseline_model,  sample_en)
aug_preds  = translate(augmented_model, sample_en)

In [ ]:
print(f"{'='*80}")
print("QUALITATIVE TRANSLATION COMPARISON (Test Set)")
print(f"{'='*80}\n")

for i in range(len(sample_en)):
    print(f"[{i+1}] Source (EN)  : {sample_en[i]}")
    print(f"    Reference (HI): {sample_hi[i]}")
    print(f"    Baseline      : {base_preds[i]}")
    print(f"    Augmented     : {aug_preds[i]}")
    print()

## Training Loss Curves

In [ ]:
def extract_loss_history(trainer):
    logs = trainer.state.log_history
    train_loss = [(e["epoch"], e["loss"]) for e in logs if "loss" in e]
    eval_loss  = [(e["epoch"], e["eval_loss"]) for e in logs if "eval_loss" in e]
    return train_loss, eval_loss

b_train_loss, b_eval_loss = extract_loss_history(baseline_trainer)
a_train_loss, a_eval_loss = extract_loss_history(augmented_trainer)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, train_loss, eval_loss, title in [
    (axes[0], b_train_loss, b_eval_loss, f"Baseline ({len(baseline_train_en)} pairs)"),
    (axes[1], a_train_loss, a_eval_loss, f"Augmented ({augmented_pair_count} pairs)"),
]:
    if train_loss:
        ax.plot(*zip(*train_loss), label="Train Loss", marker="o", markersize=3)
    if eval_loss:
        ax.plot(*zip(*eval_loss), label="Val Loss", marker="s", markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()


In [ ]:
baseline_model  = baseline_trainer.model
augmented_model = augmented_trainer.model
sample_en=["i am going to school","you wouldn't go to school"]
base_preds = translate(baseline_model,  sample_en)
aug_preds  = translate(augmented_model, sample_en)
for i in range(len(sample_en)):
    print(f"[{i+1}] Source (EN)  : {sample_en[i]}")
    print(f"    Baseline      : {base_preds[i]}")
    print(f"    Augmented     : {aug_preds[i]}")
    print()